# FruitCount — Notebook điều phối trên Kaggle

Notebook này chỉ **điều phối**: toàn bộ logic nằm trong `src/` và `scripts/`.

Trước khi chạy:
1. Upload thư mục `FruitCount/` thành một Kaggle Dataset (ví dụ `fruitcount-code`) và Add Data vào notebook.
2. Add Data bộ MinneApple (hoặc bộ đếm trái cây khác).
3. Settings → Accelerator: GPU (T4/P100), Internet: On (để tải trọng số pretrained).

Kaggle chỉ giữ `/kaggle/working` sau khi chạy xong, nên mọi kết quả được ghi vào đó.

In [ ]:
import os, shutil, sys

CODE_INPUT = '/kaggle/input/fruitcount-code/FruitCount'   # sửa theo tên dataset của bạn
WORK = '/kaggle/working/FruitCount'
if not os.path.exists(WORK):
    shutil.copytree(CODE_INPUT, WORK)
os.chdir(WORK)
sys.path.insert(0, WORK)

!pip install -q ultralytics pyyaml
!python -c "import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())"

## 1. Cấu hình dữ liệu
Sửa đường dẫn cho đúng cấu trúc bộ dữ liệu bạn Add vào (xem `/kaggle/input`).

In [ ]:
from src.utils import load_yaml, save_yaml

ds = load_yaml('configs/dataset.yaml')
ds['paths']['images_dir'] = '/kaggle/input/minneapple/detection/train/images'
ds['paths']['masks_dir']  = '/kaggle/input/minneapple/detection/train/masks'
ds['processed_dir'] = '/kaggle/working/processed'
save_yaml(ds, 'configs/dataset.yaml')
print(ds['paths'])

In [ ]:
!python scripts/prepare_data.py

## 2. Huấn luyện
Kaggle giới hạn ~9–12 giờ/phiên và có thể ngắt; `--resume` tiếp tục từ `last.pt`. Chạy từng model ở các cell/phiên riêng nếu cần.

In [ ]:
MODELS = ['yolov8', 'csrnet']          # thêm: 'yolo11', 'faster_rcnn', 'mask_rcnn'
EPOCHS = None                           # ví dụ 5 để chạy thử nhanh

for m in MODELS:
    extra = f'--epochs {EPOCHS}' if EPOCHS else ''
    !python scripts/train.py --model configs/models/{m}.yaml --out /kaggle/working/runs {extra}

## 3. Đánh giá trên tập test

In [ ]:
for m in MODELS:
    !python scripts/evaluate.py --model configs/models/{m}.yaml --out /kaggle/working/runs --split test --speed

# YOLO + tiling (SAHI-style)
!python scripts/evaluate.py --model configs/models/yolov8.yaml --out /kaggle/working/runs --split test --tile --tag _tile

In [ ]:
!python scripts/compare.py --runs /kaggle/working/runs --split test

import pandas as pd
pd.read_csv('/kaggle/working/runs/comparison/comparison_test.csv')

## 4. Phân tích: quả chồng lấn nhiều thì model nào đếm tốt hơn?

In [ ]:
from IPython.display import Image, display
display(Image('/kaggle/working/runs/comparison/error_by_count_test.png'))
display(Image('/kaggle/working/runs/comparison/comparison_test.png'))

## 5. Ví dụ minh họa (cho báo cáo)

In [ ]:
import glob, random
from src.data import load_split_records

test = load_split_records('/kaggle/working/processed', 'test')
samples = random.Random(0).sample(test, min(4, len(test)))
os.makedirs('/kaggle/working/samples', exist_ok=True)
for r in samples:
    shutil.copy(r.image_path, f'/kaggle/working/samples/{r.image_id}.png')
    print(r.image_id, 'GT =', r.count)

for m in MODELS:
    !python scripts/predict.py --model configs/models/{m}.yaml --runs /kaggle/working/runs --source /kaggle/working/samples --out /kaggle/working/vis/{m}

In [ ]:
import matplotlib.pyplot as plt, matplotlib.image as mpimg

ids = [r.image_id for r in samples]
fig, axes = plt.subplots(len(ids), len(MODELS), figsize=(6 * len(MODELS), 4 * len(ids)), squeeze=False)
for i, iid in enumerate(ids):
    for j, m in enumerate(MODELS):
        axes[i][j].imshow(mpimg.imread(f'/kaggle/working/vis/{m}/{iid}_count.jpg'))
        axes[i][j].set_title(m); axes[i][j].axis('off')
plt.tight_layout(); plt.show()